# 1. First simulation

This notebook builds a small optical bench: a HeNe laser focused by a lens on a camera. It shows the
main concepts of LUMOS:

- **elements** with physical parameters (astropy quantities) and typed ports;
- the **context**, which holds the graph of elements and computes the requested outputs;
- **figures** with physical axes, and **archives** that keep every run traceable.

In [ ]:
import numpy as np
from astropy import units as u

import lumos
from lumos import Context, plotting
from lumos.elements.detectors import Camera
from lumos.elements.optics import FreeSpace, ThinLens
from lumos.elements.sources import GaussianLaser
from lumos.fields import Grid
from lumos.plotting import style

style.apply()

## Build the bench

Elements are added to a context and connected with `>>` (output of the left element to the input of the
right one). The focal plane is sampled on its own grid: 64 × 64 pixels of 2 µm.

In [ ]:
ctx = Context(seed=42, name="first simulation")
laser = ctx.add(GaussianLaser(name="laser", wavelength=633 * u.nm, power=1 * u.mW, waist=1 * u.mm))
lens = ctx.add(ThinLens(name="lens", focal_length=200 * u.mm))
focus = ctx.add(
    FreeSpace(name="focus", distance=200 * u.mm, method="fresnel_mft", output_grid=Grid(64, dx=2 * u.um))
)
camera = ctx.add(Camera(name="camera", exposure=1 * u.us, full_well=1e9 * u.electron, bits=32))
laser >> lens >> focus >> camera
camera

In [ ]:
plotting.plot_context(ctx);

## Run and archive

`ctx.run` computes only what the requested outputs need. With `archive=...`, the run is archived in an
`archives/` directory next to this notebook: the context (`context.lumos`), the metadata (versions, git
commit, hardware, seed...), the journal and the computed outputs.

In [ ]:
result = ctx.run(["laser.out", "focus.out", "camera.out"], archive="first-simulation")
archive = result.archive
archive

In [ ]:
fig, axes = style.figure("double", ncols=3, aspect=0.3)
plotting.plot_intensity(result["laser.out"], ax=axes[0])
plotting.plot_psf(result["focus.out"], ax=axes[1], log=False)
plotting.plot_data(result["camera.out"], ax=axes[2])
axes[0].set_title("laser")
axes[1].set_title("focal plane")
axes[2].set_title("camera")
archive.save_figure(fig, "beam_and_focus");

## Compare with Gaussian optics

A Gaussian beam of waist $w_0$ focused by a lens of focal length $f$ has a waist
$w_f = \lambda f / (\pi w_0)$ in the focal plane (Siegman, *Lasers*, §17.1), so its intensity
profile is $I(r) = I_0 \exp(-2 r^2 / w_f^2)$.

In [ ]:
w_f = (633 * u.nm * 200 * u.mm / (np.pi * 1 * u.mm)).to(u.um)
fig, ax = style.figure("single")
plotting.plot_radial_profile(result["focus.out"], ax=ax, log=False, label="LUMOS")
r = np.linspace(0, 60, 200) * u.um
peak = float(result["focus.out"].intensity.max())
ax.plot(r.value, peak * np.exp(-2 * (r / w_f).decompose() ** 2), "--", label="Gaussian optics")
ax.legend()
archive.save_figure(
    fig, "radial_profile", data={"r": r, "gaussian": peak * np.exp(-2 * (r / w_f).decompose().value ** 2)}
)
w_f

## Save, reload and share

The context can be saved as a `.lumos` file, exported as an equivalent Python script, and the presets of
the web interface are such files.

In [ ]:
print(ctx.to_python())

In [ ]:
preset = lumos.io.load("presets/first_simulation.lumos")
preset.meta["description"], preset.run(preset.meta["outputs"])["camera.out"]

In [ ]:
loaded = lumos.load_archive(archive.path)
loaded.metadata["lumos_version"], sorted(loaded.data), sorted(loaded.figures)